In [1]:
import os
import random

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.pipeline import Pipeline

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_colwidth", 200)

In [2]:
DATA_PATH = "../data/processed/m_absa_selected.csv"

df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8"
)

print("Shape:", df.shape)
df.head()

,text,aspect,category,sentiment,domain,language,split,text_length,word_count
0,We have gone for dinner only a few times but the same great quality and service is given .,service,service general,positive,restaurant,en,train,90,19
1,We have gone for dinner only a few times but the same great quality and service is given .,dinner,food quality,positive,restaurant,en,train,90,19
2,"Its dark , and cozy . . there is always jazz music playing when we go .",NaN,ambience general,positive,restaurant,en,train,71,17
3,This place has great indian chinese food .,indian chinese food,food quality,positive,restaurant,en,train,42,8
4,Not what I would expect for the price and prestige of this location .,location,restaurant prices,neutral,restaurant,en,train,69,14


In [3]:
print(df["split"].value_counts())

print("\nLanguages:")
print(df["language"].value_counts())

print("\nDomains:")
print(df["domain"].value_counts())

split
train         16058
test           7611
validation     3913
Name: count, dtype: int64

Languages:
language
en    9209
ru    9190
de    9183
Name: count, dtype: int64

Domains:
domain
restaurant    9841
hotel         8877
laptop        8864
Name: count, dtype: int64


In [4]:
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (16058, 9)
Validation: (3913, 9)
Test: (7611, 9)


In [5]:
sentiment_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=SEED
        )
    )
])

In [6]:
sentiment_model.fit(
    train_df["text"],
    train_df["sentiment"]
)

print("Sentiment model trained.")

Sentiment model trained.


In [7]:
val_sentiment_pred = sentiment_model.predict(
    val_df["text"]
)

test_sentiment_pred = sentiment_model.predict(
    test_df["text"]
)

In [8]:
val_accuracy = accuracy_score(
    val_df["sentiment"],
    val_sentiment_pred
)

val_precision, val_recall, val_f1, _ = (
    precision_recall_fscore_support(
        val_df["sentiment"],
        val_sentiment_pred,
        average="macro",
        zero_division=0
    )
)

print(f"Validation Accuracy : {val_accuracy:.4f}")
print(f"Validation Precision: {val_precision:.4f}")
print(f"Validation Recall   : {val_recall:.4f}")
print(f"Validation Macro-F1 : {val_f1:.4f}")

Validation Accuracy : 0.7700
Validation Precision: 0.5629
Validation Recall   : 0.5754
Validation Macro-F1 : 0.5685


In [9]:
print(
    classification_report(
        val_df["sentiment"],
        val_sentiment_pred,
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

    negative     0.6365    0.6964    0.6651      1041
     neutral     0.1756    0.1905    0.1827       189
    positive     0.8766    0.8394    0.8576      2683

    accuracy                         0.7700      3913
   macro avg     0.5629    0.5754    0.5685      3913
weighted avg     0.7789    0.7700    0.7738      3913



In [10]:
test_accuracy = accuracy_score(
    test_df["sentiment"],
    test_sentiment_pred
)

test_precision, test_recall, test_f1, _ = (
    precision_recall_fscore_support(
        test_df["sentiment"],
        test_sentiment_pred,
        average="macro",
        zero_division=0
    )
)

print(f"Test Accuracy : {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall   : {test_recall:.4f}")
print(f"Test Macro-F1 : {test_f1:.4f}")

Test Accuracy : 0.7971
Test Precision: 0.5729
Test Recall   : 0.6054
Test Macro-F1 : 0.5858


In [11]:
cm = confusion_matrix(
    test_df["sentiment"],
    test_sentiment_pred,
    labels=["negative", "neutral", "positive"]
)

cm_df = pd.DataFrame(
    cm,
    index=["negative", "neutral", "positive"],
    columns=["negative", "neutral", "positive"]
)

cm_df

,negative,neutral,positive
negative,1220,92,346
neutral,144,71,87
positive,702,173,4776


In [12]:
train_df["category_input"] = (
    train_df["text"].fillna("")
    + " [ASPECT] "
    + train_df["aspect"].fillna("")
)

val_df["category_input"] = (
    val_df["text"].fillna("")
    + " [ASPECT] "
    + val_df["aspect"].fillna("")
)

test_df["category_input"] = (
    test_df["text"].fillna("")
    + " [ASPECT] "
    + test_df["aspect"].fillna("")
)

In [13]:
category_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1500,
            class_weight="balanced",
            random_state=SEED
        )
    )
])

In [14]:
category_model.fit(
    train_df["category_input"],
    train_df["category"]
)

print("Category model trained.")

Category model trained.


In [15]:
val_category_pred = category_model.predict(
    val_df["category_input"]
)

test_category_pred = category_model.predict(
    test_df["category_input"]
)

In [16]:
val_category_precision, val_category_recall, val_category_f1, _ = (
    precision_recall_fscore_support(
        val_df["category"],
        val_category_pred,
        average="macro",
        zero_division=0
    )
)

print(f"Validation Macro-F1: {val_category_f1:.4f}")

Validation Macro-F1: 0.1579


In [17]:
test_category_precision, test_category_recall, test_category_f1, _ = (
    precision_recall_fscore_support(
        test_df["category"],
        test_category_pred,
        average="macro",
        zero_division=0
    )
)

print(f"Test Macro-F1: {test_category_f1:.4f}")

Test Macro-F1: 0.1694


In [18]:
baseline_results = pd.DataFrame([
    {
        "task": "Sentiment",
        "model": "TF-IDF + Logistic Regression",
        "validation_macro_f1": val_f1,
        "test_macro_f1": test_f1,
        "test_accuracy": test_accuracy
    },
    {
        "task": "Aspect Category",
        "model": "TF-IDF + Logistic Regression",
        "validation_macro_f1": val_category_f1,
        "test_macro_f1": test_category_f1,
        "test_accuracy": accuracy_score(
            test_df["category"],
            test_category_pred
        )
    }
])

baseline_results

,task,model,validation_macro_f1,test_macro_f1,test_accuracy
0,Sentiment,TF-IDF + Logistic Regression,0.568486,0.585779,0.797136
1,Aspect Category,TF-IDF + Logistic Regression,0.157888,0.169377,0.375641


In [19]:
os.makedirs("../data/processed", exist_ok=True)

baseline_results.to_csv(
    "../data/processed/baseline_results.csv",
    index=False
)

In [20]:
language_results = []

for language in ["en", "ru", "de"]:

    train_lang = train_df[
        train_df["language"] == language
    ]

    test_lang = test_df[
        test_df["language"] == language
    ]

    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True
            )
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=SEED
            )
        )
    ])

    model.fit(
        train_lang["text"],
        train_lang["sentiment"]
    )

    predictions = model.predict(
        test_lang["text"]
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            test_lang["sentiment"],
            predictions,
            average="macro",
            zero_division=0
        )
    )

    accuracy = accuracy_score(
        test_lang["sentiment"],
        predictions
    )

    language_results.append({
        "language": language,
        "accuracy": accuracy,
        "macro_f1": f1,
        "precision": precision,
        "recall": recall
    })

language_results = pd.DataFrame(language_results)

language_results

,language,accuracy,macro_f1,precision,recall
0,en,0.811588,0.606086,0.590760,0.629337
1,ru,0.773318,0.558607,0.544556,0.582170
2,de,0.807738,0.595777,0.590867,0.608342


In [21]:
domain_results = []

for domain in ["restaurant", "hotel", "laptop"]:

    train_domain = train_df[
        train_df["domain"] == domain
    ]

    test_domain = test_df[
        test_df["domain"] == domain
    ]

    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True
            )
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=SEED
            )
        )
    ])

    model.fit(
        train_domain["text"],
        train_domain["sentiment"]
    )

    predictions = model.predict(
        test_domain["text"]
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            test_domain["sentiment"],
            predictions,
            average="macro",
            zero_division=0
        )
    )

    accuracy = accuracy_score(
        test_domain["sentiment"],
        predictions
    )

    domain_results.append({
        "domain": domain,
        "accuracy": accuracy,
        "macro_f1": f1,
        "precision": precision,
        "recall": recall
    })

domain_results = pd.DataFrame(domain_results)

domain_results

,domain,accuracy,macro_f1,precision,recall
0,restaurant,0.738314,0.548124,0.568201,0.561407
1,hotel,0.854273,0.517244,0.513066,0.523697
2,laptop,0.742757,0.585494,0.578335,0.596427


In [22]:
language_results.to_csv(
    "../data/processed/baseline_sentiment_by_language.csv",
    index=False
)

domain_results.to_csv(
    "../data/processed/baseline_sentiment_by_domain.csv",
    index=False
)

In [23]:
os.makedirs("../data/processed", exist_ok=True)

baseline_results.to_csv(
    "../data/processed/baseline_results.csv",
    index=False
)

language_results.to_csv(
    "../data/processed/baseline_sentiment_by_language.csv",
    index=False
)

domain_results.to_csv(
    "../data/processed/baseline_sentiment_by_domain.csv",
    index=False
)

print("Baseline results saved successfully.")

Baseline results saved successfully.


In [24]:
import joblib

os.makedirs("../models/baseline", exist_ok=True)

joblib.dump(
    sentiment_model,
    "../models/baseline/sentiment_tfidf_logreg.joblib"
)

joblib.dump(
    category_model,
    "../models/baseline/category_tfidf_logreg.joblib"
)

print("Models saved successfully.")

Models saved successfully.
